# Stage 5 — Interpretability Evaluation

**Goal:** Connect CTLS to the existing mechanistic interpretability literature
via SAE-based monosemanticity analysis. Provide quantitative evidence that CTLS
training produces more interpretable internal representations.

**Method:**
1. Train one Sparse Autoencoder (SAE) per layer on collected activations
2. Analyse learned features for monosemanticity — does each feature fire for
   one class (good) or many (superposition)?
3. Compare CTLS vs Baseline across three metrics:
   - **Monosemanticity score** — fraction of features dominated by a single class
   - **Feature entropy** — entropy of per-class activation distribution (lower = more monosemantic)
   - **Circuit reuse rate** — fraction of features active for >1 class

**Why this matters:** SAE analysis is the dominant evaluation methodology in the
mechanistic interpretability community (Anthropic's sparse features work, etc.).
Showing improvement on these metrics bridges CTLS to that literature.

**Compute note:** Training 8 SAEs (one per ResNet18 block) × 2 models = 16 SAE
training runs. Each takes 1–3 min on GPU. Budget ~30–45 min total.

## 0. Setup

In [ ]:
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = 'https://github.com/jacobposchl/model-interpretability'  # <-- edit this once
    REPO_DIR = '/content/ctls'
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {REPO_URL} {REPO_DIR}')
        os.system(f'pip install -r {REPO_DIR}/requirements.txt -q')
    ROOT = REPO_DIR
else:
    ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print(f'Working directory: {os.getcwd()} ({"Colab" if IN_COLAB else "local"})')


In [ ]:
from models.soft_mask import SoftMask
from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder
from data.cifar import get_standard_loaders

def load_model(config_path, checkpoint_path, device):
    with open(config_path) as f:
        cfg = yaml.safe_load(f)
    mcfg = cfg['model']
    ecfg = mcfg['meta_encoder']
    tcfg = cfg['training']
    soft_mask = SoftMask(init_temperature=tcfg['temperature']['init']).to(device)
    backbone = CTLSBackbone(
        arch=mcfg['arch'], num_classes=mcfg['num_classes'],
        soft_mask=soft_mask, pretrained=False,
    ).to(device)
    meta_encoder = MetaEncoder(
        layer_dims=backbone.layer_dims,
        hidden_dim=ecfg.get('hidden_dim', 256),
        embedding_dim=ecfg.get('embedding_dim', 64),
        encoder_type=ecfg.get('encoder_type', 'mlp'),
    ).to(device)
    ckpt = torch.load(checkpoint_path, map_location=device)
    backbone.load_state_dict(ckpt['backbone_state'])
    meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
    backbone.eval()
    meta_encoder.eval()
    return backbone, meta_encoder, cfg

bb_ctls, me_ctls, config = load_model('configs/ctls.yaml', 'experiments/ctls/best.pt', DEVICE)
bb_base, me_base, _ = load_model('configs/baseline.yaml', 'experiments/baseline/best.pt', DEVICE)

dcfg = config['data']
_, val_loader = get_standard_loaders(
    data_dir=dcfg['data_dir'], batch_size=dcfg['batch_size'],
    num_workers=dcfg.get('num_workers', 4), augment=False, download=True,
)
print('Models and data loaded.')

## 1. SAE Hyperparameters

Key choices:
- `dict_size_multiplier=4` — SAE dictionary is 4× wider than input dim (overcomplete)
- `l1_coeff=1e-3` — sparsity pressure. Higher = sparser features, but worse reconstruction.
  Tune if features are either all-zero or all-active.
- `mono_threshold=0.8` — a feature is "monosemantic" if one class contributes >80%
  of its total activation across the val set

In [ ]:
SAE_EPOCHS         = 50
N_SAMPLES          = 10000
DICT_SIZE_MULT     = 4
L1_COEFF           = 1e-3
MONO_THRESHOLD     = 0.8
ACTIVE_THRESHOLD   = 0.01

## 2. Run Full Monosemanticity Analysis (CTLS vs Baseline)

This trains 16 SAEs and computes all metrics. It's the main compute cell.

In [ ]:
from evaluation.monosemanticity import MonosemanticityScorer

scorer = MonosemanticityScorer(
    backbone=bb_ctls,
    loader=val_loader,
    device=DEVICE,
    dict_size_multiplier=DICT_SIZE_MULT,
    l1_coeff=L1_COEFF,
    mono_threshold=MONO_THRESHOLD,
    active_threshold=ACTIVE_THRESHOLD,
)

print('Running monosemanticity comparison (CTLS vs Baseline)...')
print('This trains one SAE per layer per model — expect ~30–45 min on GPU.\n')

comparison = scorer.compare_with_baseline(
    baseline_backbone=bb_base,
    sae_epochs=SAE_EPOCHS,
    n_samples=N_SAMPLES,
)

print('\n=== Done ===')

## 3. Results Table

In [ ]:
print('=== Monosemanticity: CTLS vs Baseline ===')
print(f"{'Layer':>6} | {'Base mono':>10} {'CTLS mono':>10} {'Δmono':>8} | "
      f"{'Base reuse':>11} {'CTLS reuse':>11} {'Δreuse':>8}")
print('-' * 80)

for row in comparison['layer_results']:
    l = row['layer_idx'] + 1
    print(f"{l:>6} | "
          f"{row['base_mono']:>10.3f} {row['ctls_mono']:>10.3f} {row['delta_mono']:>+8.3f} | "
          f"{row['base_reuse']:>11.3f} {row['ctls_reuse']:>11.3f} {row['delta_reuse']:>+8.3f}")

# Summary averages
rows = comparison['layer_results']
avg_delta_mono  = np.mean([r['delta_mono']  for r in rows])
avg_delta_reuse = np.mean([r['delta_reuse'] for r in rows])
print('-' * 80)
print(f"{'Average':>6} |            {'':>10} {avg_delta_mono:>+8.3f} |             {'':>11} {avg_delta_reuse:>+8.3f}")
print()
print(f'Interpretation:')
print(f'  Δmono > 0  means CTLS has more monosemantic features (better)')
print(f'  Δreuse < 0 means CTLS has less feature sharing across classes (better)')

## 4. Visualise: Monosemanticity and Reuse Rate by Layer

In [ ]:
rows = comparison['layer_results']
layers = [r['layer_idx'] + 1 for r in rows]

mono_base  = [r['base_mono']  for r in rows]
mono_ctls  = [r['ctls_mono']  for r in rows]
reuse_base = [r['base_reuse'] for r in rows]
reuse_ctls = [r['ctls_reuse'] for r in rows]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Stage 5: SAE-based monosemanticity — CTLS vs Baseline', fontsize=13)

ax1.plot(layers, mono_base, 'o--', color='steelblue', lw=2, label='Baseline')
ax1.plot(layers, mono_ctls, 's-',  color='darkorange', lw=2.5, label='CTLS')
ax1.set_xlabel('Layer index')
ax1.set_ylabel('Monosemanticity score')
ax1.set_title(f'Fraction of features with >80% single-class activation')
ax1.set_xticks(layers)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(layers, reuse_base, 'o--', color='steelblue', lw=2, label='Baseline')
ax2.plot(layers, reuse_ctls, 's-',  color='darkorange', lw=2.5, label='CTLS')
ax2.set_xlabel('Layer index')
ax2.set_ylabel('Circuit reuse rate')
ax2.set_title(f'Fraction of features active for >1 class (superposition)')
ax2.set_xticks(layers)
ax2.legend()
ax2.grid(True, alpha=0.3)

fig.tight_layout()
os.makedirs('experiments/ctls', exist_ok=True)
fig.savefig('experiments/ctls/stage5_monosemanticity.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Entropy Distribution

For each layer, plot the distribution of per-feature entropy. Lower entropy =
more monosemantic. We want CTLS to push the distribution leftward.

In [ ]:
# Extract per-feature entropy from the full results
ctls_results = comparison['ctls']
base_results = comparison['baseline']

entropies_ctls = [r['mean_feature_entropy'] for r in ctls_results]
entropies_base = [r['mean_feature_entropy'] for r in base_results]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(layers, entropies_base, 'o--', color='steelblue', lw=2, label='Baseline')
ax.plot(layers, entropies_ctls, 's-',  color='darkorange', lw=2.5, label='CTLS')
ax.set_xlabel('Layer index')
ax.set_ylabel('Mean feature entropy (nats)')
ax.set_title('Per-layer mean SAE feature entropy (lower = more monosemantic)')
ax.set_xticks(layers)
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('experiments/ctls/stage5_entropy.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Per-Class Monosemanticity Heatmap (Final Layer)

For the final layer (most semantic), visualise which classes have the most
monosemantic SAE features. Some classes may be naturally easier to separate
(e.g. ship, airplane) vs. harder (cat, dog, deer).

In [ ]:
from evaluation.monosemanticity import SparseAutoencoder, SAETrainer
import torch.nn.functional as F

CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer',
                   'dog','frog','horse','ship','truck']

FINAL_LAYER = len(bb_ctls.layer_dims) - 1

def collect_layer_acts(backbone, loader, layer_idx, n=10000):
    all_acts, all_labels = [], []
    total = 0
    with torch.no_grad():
        for batch in loader:
            x, labels = batch
            x = x.to(DEVICE)
            _, traj = backbone(x)
            all_acts.append(traj[layer_idx].cpu())
            all_labels.append(labels)
            total += x.shape[0]
            if total >= n: break
    return torch.cat(all_acts, 0)[:n], torch.cat(all_labels, 0)[:n]

def train_sae(acts, input_dim, dict_size, l1_coeff, sae_epochs, device):
    sae = SparseAutoencoder(input_dim, dict_size, l1_coeff).to(device)
    trainer = SAETrainer(sae, device)
    trainer.train_on_activations(acts, epochs=sae_epochs)
    return sae

# Train SAE on final layer for both models
print('Training SAE on final layer (CTLS)...')
acts_ctls, labels = collect_layer_acts(bb_ctls, val_loader, FINAL_LAYER)
input_dim = acts_ctls.shape[-1]
sae_ctls = train_sae(acts_ctls, input_dim, input_dim * DICT_SIZE_MULT, L1_COEFF, SAE_EPOCHS, DEVICE)

print('Training SAE on final layer (Baseline)...')
acts_base, _ = collect_layer_acts(bb_base, val_loader, FINAL_LAYER)
sae_base = train_sae(acts_base, input_dim, input_dim * DICT_SIZE_MULT, L1_COEFF, SAE_EPOCHS, DEVICE)

print('Done.')

In [ ]:
def class_feature_matrix(sae, acts, labels, num_classes=10):
    """Returns [num_classes, dict_size] matrix of mean activations per class."""
    sae.eval()
    with torch.no_grad():
        _, features = sae(acts.to(DEVICE))
        features = features.cpu().numpy()
    labels_np = labels.numpy()
    matrix = np.zeros((num_classes, features.shape[1]))
    for cls in range(num_classes):
        mask = labels_np == cls
        if mask.sum() > 0:
            matrix[cls] = features[mask].mean(axis=0)
    return matrix

mat_ctls = class_feature_matrix(sae_ctls, acts_ctls, labels)
mat_base = class_feature_matrix(sae_base, acts_base, labels)

# Sort features by max-class for visualisation
sort_ctls = np.argsort(mat_ctls.argmax(axis=0))
sort_base = np.argsort(mat_base.argmax(axis=0))

# Show top 200 features (most active)
top_k = 200
active_ctls = np.argsort(mat_ctls.max(axis=0))[-top_k:][::-1]
active_base = np.argsort(mat_base.max(axis=0))[-top_k:][::-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Stage 5: SAE feature selectivity at layer {FINAL_LAYER+1} (top {top_k} features)', fontsize=13)

for ax, mat, active, title in [
    (axes[0], mat_base, active_base, 'Baseline'),
    (axes[1], mat_ctls, active_ctls, 'CTLS'),
]:
    # Sort selected features by dominant class
    sel = mat[:, active]
    sort_idx = np.argsort(sel.argmax(axis=0))
    im = ax.imshow(sel[:, sort_idx], aspect='auto', cmap='hot', vmin=0)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('SAE feature (sorted by dominant class)')
    ax.set_yticks(range(10))
    ax.set_yticklabels(CIFAR10_CLASSES, fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.8, label='Mean activation')

fig.tight_layout()
fig.savefig('experiments/ctls/stage5_feature_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: experiments/ctls/stage5_feature_heatmap.png')
print('\nBlocks of clean colour per row = monosemantic features')
print('Mixed colour per row = polysemantic (superposition) features')

## 7. SAE Reconstruction Quality

Check that the SAEs aren't just trivially reconstructing — higher L1 sparsity
pressure can degrade reconstruction. Track reconstruction MSE alongside monosemanticity.

In [ ]:
def recon_mse(sae, acts):
    sae.eval()
    with torch.no_grad():
        x = acts.to(DEVICE)
        recon, features = sae(x)
        mse = F.mse_loss(recon, x).item()
        sparsity = (features > 0).float().mean().item()
    return mse, sparsity

mse_ctls, sp_ctls = recon_mse(sae_ctls, acts_ctls)
mse_base, sp_base = recon_mse(sae_base, acts_base)

print('=== SAE Quality Check ===')
print(f"{'':20} {'Recon MSE':>12} {'Sparsity':>12}")
print(f"{'CTLS SAE':<20} {mse_ctls:>12.5f} {sp_ctls:>12.4f}")
print(f"{'Baseline SAE':<20} {mse_base:>12.5f} {sp_base:>12.4f}")
print()
print('Sparsity = fraction of SAE features active on average')
print('Target: sparsity < 0.2 (most features off = sparse = good)')

## Summary

Fill in the final results table:

| Metric (avg over layers) | Baseline | CTLS | Delta |
|--------------------------|---------|------|-------|
| Monosemanticity score | ___ | ___ | ___ |
| Mean feature entropy | ___ | ___ | ___ |
| Circuit reuse rate | ___ | ___ | ___ |
| Final-layer recon MSE (SAE) | ___ | ___ | ___ |

**Interpreting the results:**
- CTLS mono > Baseline mono → CTLS representations are more monosemantic ✓
- CTLS entropy < Baseline entropy → features more class-selective ✓
- CTLS reuse < Baseline reuse → less superposition pressure ✓
- Heatmap: cleaner per-row blocks for CTLS → visually more structured circuits ✓

---

**This completes the five-stage proof of concept.** 

With all five stages done and results in hand, the paper's core claims can be evaluated:
1. CTLS produces more structured circuit spaces (Stage 2 silhouette)
2. Circuit space ≠ output space (Stage 3 distances)
3. Depth-weighting is a meaningful design choice (Stage 4 ablation)
4. CTLS produces more interpretable representations by SAE metrics (Stage 5)